# Tarefa 03

- Leia os enunciados com atenção
- Saiba que pode haver mais de uma resposta correta
- Insira novas células de código sempre que achar necessário
- Em caso de dúvidas, procure os Tutores
- Divirta-se :)

In [2]:
import pandas as pd
import requests

####  1) Lendo de APIs
Vimos em aula como carregar dados públicos do governo através de um API (*Application Programming Interface*). No exemplo de aula, baixamos os dados de pedidos de verificação de limites (PVL) realizados por estados, e selecionamos apenas aqueles referentes ao estado de São Paulo.

1. Repita os mesmos passos feitos em aula, mas selecione os PVLs realizados por municípios no estado do Rio de Janeiro.
2. Quais são os três *status* das solicitações mais frequentes na base? Quais são suas frequências?
3. Construa uma nova variável que contenha o ano do **status**. Observe que ```data_status``` vem como tipo *object* no **DataFrame**. Dica: você pode usar o método ```.str``` para transformar o tipo da variável em string, em seguida um método como [**slice()**](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.slice.html) ou [**split()**](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.Series.str.split.html).
4. Indique a frequência de cada ano do campo construído no item (3).

In [3]:
# 1) Seu código aqui
url = 'https://apidatalake.tesouro.gov.br/ords/sadipem/tt/pvl'
r = requests.get(url)
r.raise_for_status() #verificar erro na requisição

data = r.json()

In [4]:
df = pd.DataFrame(data['items'])
# print(df.head())
# print(df.columns)
df_rj = df[df['uf'] == 'RJ']
print(df_rj.head())
    
    

     id_pleito tipo_interessado      interessado  cod_ibge  uf num_pvl  \
86       11576        Município           Quatis   3304128  RJ    None   
274      12505        Município    Volta Redonda   3306305  RJ    None   
278      11447        Município     Belford Roxo   3300456  RJ    None   
335      11343        Município  Duque de Caxias   3301702  RJ    None   
348      13080        Município        Três Rios   3306008  RJ    None   

           status          num_processo        data_protocolo  \
86   Regularizado  17944.001480/2009-15  2012-05-07T00:00:00Z   
274  Regularizado  17944.001701/2011-70  2013-12-04T00:00:00Z   
278      Deferido  17944.001450/2008-28  2008-06-10T00:00:00Z   
335  Regularizado  17944.001416/2006-91  2007-02-28T00:00:00Z   
348      Deferido  17944.001905/2014-53  2014-12-18T00:00:00Z   

                   tipo_operacao                                  finalidade  \
86   Operação contratual interna  Regularização de Dívida - Energia Elétrica   
274 

In [5]:
print(df.columns)

Index(['id_pleito', 'tipo_interessado', 'interessado', 'cod_ibge', 'uf',
       'num_pvl', 'status', 'num_processo', 'data_protocolo', 'tipo_operacao',
       'finalidade', 'tipo_credor', 'credor', 'moeda', 'valor',
       'pvl_assoc_divida', 'pvl_contradado_credor', 'data_status'],
      dtype='object')


In [6]:
# 2) Seu código aqui
# df_rj.columns
status_frequencia = df_rj['status'].value_counts()
top_3_status = status_frequencia.head(3)
#número total de solicitações
total_solicitacoes_rj = len(df_rj)
#porcentagem de cada status
status_percentual = (status_frequencia / total_solicitacoes_rj) * 100
top_3_status_percentual = (top_3_status / total_solicitacoes_rj) * 100

df_top_3_completo = pd.DataFrame({
    'Contagem': top_3_status,
    'Percentual (%)': top_3_status_percentual
})
print(df_top_3_completo)

                                                    Contagem  Percentual (%)
status                                                                      
Deferido                                                  29       37.179487
Arquivado                                                 15       19.230769
Encaminhado à PGFN com manifestação técnica fav...        12       15.384615


In [8]:
# 3) Seu código aqui
# checar se há valores nulos em 'data_'
print(f' valores nulos de data_status: {df_rj['data_status'].isnull().sum()}')

 valores nulos de data_status: 0


In [9]:

# criar uma cópia
df_rj_copy = df_rj.copy()

In [10]:
def extrair_ano_da_string(data_str):
    #se for nan ou não for str retorna none
    if pd.isna(data_str) or not isinstance(data_str, str):
        return None
    try:
        #ano ultimos 4 caracteres
        ano = data_str[-4:]
        return int(ano)
    except ValueError:
        print(f"Erro ao extrair ano da string: {data_str}")
        return None

In [25]:
# Garantir que a coluna 'data_status' seja do tipo string
df_rj_copy['data_status_str'] = df_rj_copy['data_status'].astype(str)

# # aplicar função para extrair o ano da string
df_rj_copy['data_status_ano'] = df_rj_copy['data_status_str'].apply(extrair_ano_da_string)

print("\nDataFrame com a nova coluna 'data_status' (extraída via string):")
df_rj_copy[['data_status', 'data_status_str', 'data_status_ano']].head()


DataFrame com a nova coluna 'data_status' (extraída via string):


,data_status,data_status_str,data_status_ano
86,29/06/2012,29/06/2012,2012
274,14/02/2014,14/02/2014,2014
278,10/06/2008,10/06/2008,2008
335,13/03/2007,13/03/2007,2007
348,07/01/2015,07/01/2015,2015


In [ ]:
# 4)
# frequência dos itens por ano na coluna criada no exercício 3
df_frequencia_ano = df_rj_copy['data_status_ano'].value_counts().reset_index()
df_frequencia_ano.columns = ['ano', 'frequencia']
df_frequencia_ano


,ano,frequencia
0,2007,12
1,2008,11
2,2012,6
3,2023,6
4,2014,6
5,2013,5
6,2011,4
7,2010,4
8,2009,3
9,2006,3


####  2) Melhorando a interação com o API
Observe dois URLs de consultas diferentes, por exemplo o URL utilizado em aula, e o URL feito no exercício anterior. Compare-os e observe as diferenças.

1. Faça uma função em Python que recebe como argumento o UF da consulta e o tipo de interessado (```'Estado'```ou ```Município```), e que devolve os dados da consulta no formato *DataFrame*.
2. Quantas solicitações para o Estado podem ser consultadas para Minas Gerais com *status* em 'Arquivado por decurso de prazo' estão registradas?
3. Qual é o município da Bahia com mais solicitações deferidas?
4. Salve um arquivo .csv com os dados de solicitações da Bahia, com interessado = 'Estado'

In [58]:
#1) Seu código aqui
# url_1 = 'https://apidatalake.tesouro.gov.br/ords/sadipem/tt/pvl'
# url_2 = 'http://apidatalake.tesouro.gov.br/ords/sadipem/tt/pvl?uf=SP&tipo_interessado=Estado'

def consultar_dados_pvl(uf: str, tipo_interessado: str) -> pd.DataFrame:
    """
    Consulta os dados de PVL para um determinado UF e tipo de interessado.

    Args:
        uf (str): Sigla da Unidade Federativa (ex: 'SP', 'MG').
        tipo_interessado (str): Tipo de interessado ('Estado' ou 'Município').

    Returns:
        pd.DataFrame: DataFrame com os dados da consulta ou um DataFrame vazio em caso de erro.
    """
    base_url = 'http://apidatalake.tesouro.gov.br/ords/sadipem/tt/pvl'
    params = {
        'uf': uf.upper(),
        'tipo_interessado': tipo_interessado.lower().capitalize()
    }

    if not uf.isalpha() or len(uf) != 2:
        print("UF inválida. Deve conter exatamente 2 letras.")
        return pd.DataFrame()

    if tipo_interessado.lower() not in ['estado', 'município']:
        print("Tipo de interessado inválido. Deve ser 'Estado' ou 'Município'.")
        print("Cheque se município possuir acento (Município)")
        return pd.DataFrame()
    
    try:
        response = requests.get(base_url, params=params)
        response.raise_for_status() # Levanta um erro para respostas HTTP 4xx/5xx
        
        data_json = response.json()
        
        # A API do Tesouro geralmente retorna os dados dentro de uma chave 'items'
        if 'items' in data_json:
            df = pd.DataFrame(data_json['items'])
            return df
        else:
            print("A chave 'items' não foi encontrada no JSON de resposta.")
            return pd.DataFrame() # Retorna DataFrame vazio se 'items' não existir

    except requests.exceptions.HTTPError as http_err:
        print(f"Erro HTTP: {http_err}")
        print(f"URL da requisição: {response.url}")
        print(f"Conteúdo da resposta: {response.text}")
    except requests.exceptions.ConnectionError as conn_err:
        print(f"Erro de Conexão: {conn_err}")
    except requests.exceptions.Timeout as timeout_err:
        print(f"Erro de Timeout: {timeout_err}")
    except requests.exceptions.RequestException as req_err:
        print(f"Erro na Requisição: {req_err}")
    except ValueError as json_err: # Erro ao decodificar JSON
        print(f"Erro ao decodificar JSON: {json_err}")
        print(f"Conteúdo da resposta: {response.text}")
        
    return pd.DataFrame() # Retorna DataFrame vazio em caso de qualquer erro

df_mg = consultar_dados_pvl('mg', 'Estado')
df_mg.head()

,id_pleito,tipo_interessado,interessado,cod_ibge,uf,num_pvl,status,num_processo,data_protocolo,tipo_operacao,finalidade,tipo_credor,credor,moeda,valor,pvl_assoc_divida,pvl_contradado_credor,data_status
0,9197,Estado,Minas Gerais,31,MG,None,Encaminhado à PGFN com manifestação técnica fa...,17944.000924/2009-03,2010-03-10T00:00:00Z,Operação contratual externa (com garantia da U...,Infraestrutura,Instituição Financeira Internacional,Banco Interamericano de Desenvolvimento,Dólar dos EUA,50000000.0,1,0,01/04/2010
1,11696,Estado,Minas Gerais,31,MG,None,Arquivado a pedido,17944.001504/2014-01,2015-04-13T00:00:00Z,Operação contratual externa (com garantia da U...,Segurança pública,Instituição Financeira Internacional,Banco Interamericano de Desenvolvimento,Dólar dos EUA,70000000.0,0,0,29/04/2016
2,13441,Estado,Minas Gerais,31,MG,None,Deferido,19405.000001/2005-42,2005-08-16T00:00:00Z,Operação contratual interna,Infraestrutura,Instituição Financeira Nacional,Banco do Nordeste do Brasil S/A,Dólar dos EUA,27534000.0,1,0,29/08/2005
3,7032,Estado,Minas Gerais,31,MG,None,Encaminhado à PGFN com manifestação técnica fa...,17944.000482/2012-92,2012-10-11T00:00:00Z,Operação contratual externa (com garantia da U...,Reestruturação e recomposição do principal de ...,Instituição Financeira Internacional,Banco Internacional para Reconstrução e Desenv...,Dólar dos EUA,450000000.0,1,0,15/10/2012
4,13641,Estado,Minas Gerais,31,MG,None,Deferido,19405.000042/2004-58,2004-11-26T00:00:00Z,Operação contratual interna,Infraestrutura,Instituição Financeira Nacional,Banco Nacional de Desenvolvimento Econômico e ...,Real,53770000.0,1,0,08/12/2004


In [55]:
# 2) Seu código aqui
# 2. Quantas solicitações para o Estado podem ser consultadas para Minas Gerais com *status* em 'Arquivado por decurso de prazo' estão registradas?
# print(df_mg['status'].value_counts())
arquivado_por_decurso_de_prazo = df_mg[df_mg['status'] == 'Arquivado por decurso de prazo'].shape[0]
print(f'Arquivado por decurso de prazo: {arquivado_por_decurso_de_prazo}')

Arquivado por decurso de prazo: 1


In [80]:
# 3) Seu código aqui
df_ba_municipio = consultar_dados_pvl('ba', 'município')

In [81]:
solicitacoes_deferidas_ba = df_ba_municipio[df_ba_municipio['status'] == 'Deferido']

contagem_deferidas_por_municipio = solicitacoes_deferidas_ba['interessado'].value_counts()
# print(contagem_deferidas_por_municipio)

municipio_mais_deferido = contagem_deferidas_por_municipio.idxmax()
# print(municipio_mais_deferido)

numero_max_deferidas = contagem_deferidas_por_municipio.max()

print(f"O município com o maior número de solicitações deferidas é: {municipio_mais_deferido} com {numero_max_deferidas} solicitações.")

O município com o maior número de solicitações deferidas é: Luís Eduardo Magalhães com 16 solicitações.


In [ ]:
# 4) Seu código aqui
df_ba_estado = consultar_dados_pvl('BA', 'estado')
if not df_ba_estado.empty:
    nome_arquivo_csv = 'solicitacoes_bahia_estado.csv'
    try:
        df_ba_estado.to_csv(nome_arquivo_csv, index=False, sep=';', encoding='utf-8')
        print(f'O arquivo {nome_arquivo_csv} foi criado com sucesso.')
    except Exception as e:
        print(f'Ocorreu um erro ao criar o arquivo {nome_arquivo_csv}: {e}')
else:
    print('Não foram encontradas solicitações para o estado de Bahia.')